# Decision Negotiation Agent – Multi-Agent Decision Intelligence System

This project is our capstone for Google’s 5-Day AI Agents Intensive.  
It is a **multi-agent decision intelligence system** that helps a user make tough choices
(e.g., “accept a promotion vs keep a flexible role”) by letting multiple specialized
agents debate, then letting an LLM supervisor make a final call.

## Core Idea

Instead of asking a single LLM for advice, this system simulates a *mini internal committee*:

- 🧠 **LogicAgent** – optimizes time, money, and short-term productivity.
- ❤️ **EmotionAgent** – cares about stress, relief, and emotional well-being.
- 🕰 **LongTermAgent** – looks at future regret and long-term gain.
- 🎯 **ValuesAgent** – checks alignment with core life values (career, health, relationships, finances, peace).

Each agent scores the options, explains its reasoning, and casts a vote.
A **Supervisor LLM** then reads all proposals and returns a structured JSON
decision: final choice, reasoning, and a vote breakdown.

---

## System Architecture

1. **Decision Input Layer**
   - User provides:
     - `question` – the dilemma (e.g., *“Should I accept a promotion that doubles my hours?”*)
     - `options` – list of possible actions
     - optional `context` – stress level, financial pressure, priorities, etc.

2. **Specialized Agents**
   - Each agent implements a `decide(decision)` interface.
   - Returns:
     ```json
     {
       "agent_name": "...",
       "preferred_option": "...",
       "scores": { "Option A": {...}, "Option B": {...} },
       "explanation": "Natural language reasoning..."
     }
     ```

3. **LLM Supervisor (Gemini via ADK)**
   - Ingests all agent proposals.
   - Outputs **strict JSON**:
     ```json
     {
       "final_decision": "...",
       "reasoning": "...",
       "agent_votes": {
         "EmotionAgent": "...",
         "LogicAgent": "...",
         "LongTermAgent": "...",
         "ValuesAgent": "..."
       }
     }
     ```

4. **Evaluation & Observability**
   - Inspired by Day 4 of the AI Agents Intensive, the notebook includes:
     - Ambiguous / complex / edge-case test scenarios
     - Debug prints of agent proposals
     - Supervisor raw JSON
     - Structured evaluation objects for analysis

---

## Interpretability & Analytics Features

To make the agent system **explainable** and **auditable**, I added:

### ⚔️ Agent Conflict Analyzer
- Computes agreement/disagreement between agents.
- Outputs pairwise relationships and an overall **conflict score (0–1)**:
  - `0.0` → perfect agreement
  - higher values → more internal conflict

### 📈 Decision Stability Score (0–100)
- Aggregates:
  - agent consensus,
  - score variance across options,
  - supervisor–agent agreement.
- Produces a single **stability score**:
  - `> 85` → very stable decision
  - `70–85` → moderately stable
  - `< 50` → unstable / needs review

### ⚔️ Option Battle Card (Side-by-Side Comparison)
- Builds a table comparing each option across all agents:
  - emotional_relief / stress_risk
  - productivity / time_cost / money_impact
  - future_regret / future_gain
  - values alignment (career, relationships, health, finances, peace)
- Shows:
  - Agent-by-agent scores per option
  - Vote counts
  - Final winner option

This looks and feels like a **consulting-grade analysis dashboard** for decisions.

### 🎴 Agent Personality Cards
- Each agent has a defined **persona**:
  - tagline, strengths, weaknesses, and personality description.
- Cards make the system more interpretable and human-readable
  (e.g., *“LongTermAgent – forward-thinking, strategic, cautious about regret”*).

---

## Example Scenario

**Question:**  
> “Should I accept the promotion that doubles my work hours but increases salary,
> or stay in my current role with less pay but more flexibility?”

- All four agents independently analyze the trade-offs.
- The supervisor reviews their proposals and returns:
  - `final_decision`: *“Accept the promotion with doubled hours and higher salary”* (for this test context)
  - A natural language explanation merging emotional, logical, long-term, and values-based reasoning.
  - A full vote breakdown and stability score.

This demonstrates the system’s ability to:
- model trade-offs,
- surface internal disagreements,
- and justify its recommendation in an auditable way.

---

## Technical Highlights

- Built on **Gemini + ADK** patterns from Google’s 5-Day AI Agents Intensive.
- Uses:
  - custom scoring logic for each agent,
  - a supervisor LLM for final arbitration,
  - structured JSON outputs for downstream evaluation,
  - multi-scenario test harness (ambiguous / invalid / complex decisions),
  - interpretable analytics (conflict, stability, battle cards, personas).

This project shows my ability to:
- design **multi-agent LLM systems**,
- build **evaluation & observability tooling**,
- and present **complex AI behavior in a way that non-technical stakeholders can understand**.


## 🧩 System Architecture (High-Level Diagram)

                      +-------------------------+
                      |   User Decision Input   |
                      +-----------+-------------+
                                  |
                                  v
                    +----------------------------+
                    |      Specialized Agents     |
                    +----------------------------+
             /---------+---------+---------+---------\
             v                   v                   v                   v
    +----------------+  +----------------+  +----------------+  +----------------+
    |  EmotionAgent  |  |  LogicAgent    |  | LongTermAgent |  |  ValuesAgent   |
    |  - stress       |  | - productivity|  | - regret       |  | - career       |
    |  - relief       |  | - time cost   |  | - future gain  |  | - relationships |
    +----------------+  +----------------+  +----------------+  +----------------+
             \___________        |       _________ ___________/
                                 v
                  +-------------------------------+
                  |    Supervisor LLM (Gemini)    |
                  +-------------------------------+
                            |       |       |
                            v       v       v
                 +--------------+  +----------------+  +-----------------+
                 | Final JSON   |  | Reasoning Text|  | Agent Vote Map  |
                 +--------------+  +----------------+  +-----------------+
                            \         |        /
                             \        |       /
                              v       v      v
                    +----------------------------------+
                    |       Observability Layer        |
                    +----------------------------------+
                    | Conflict Analysis  | Stability   |
                    | Option Battle Card | Statistics  |
                    +----------------------------------+



In [108]:
# Interactive User Decision Runner

def run_interactive_decision():
    print("🧠 AI Decision Assistant — Interactive Mode")
    print("------------------------------------------\n")
    
    # Collect question
    question = input("Enter your decision question:\n> ")

    # Collect options (two-options system)
    option1 = input("\nEnter OPTION 1:\n> ")
    option2 = input("\nEnter OPTION 2:\n> ")

    # Optional context
    print("\n(Optional) Provide context (press Enter to skip):")
    stress = input("Stress level (0–10): ") or "5"
    money = input("Financial pressure (low/medium/high): ") or "medium"
    energy = input("Energy level (0–10): ") or "5"

    context = {
        "stress_level": float(stress),
        "financial_pressure": money,
        "energy_level": float(energy)
    }

    # Build decision object
    decision = {
        "question": question,
        "options": [option1, option2],
        "context": context
    }

    print("\n📌 Running decision through agents...\n")

    # Generate proposals
    proposals = [
        emotion_agent.decide(decision),
        logic_agent.decide(decision),
        longterm_agent.decide(decision),
        values_agent.decide(decision)
    ]

    print("🔍 Agent Proposals:")
    for p in proposals:
        print(p, "\n")

    # Supervisor LLM
    supervisor_output = run_supervisor_llm(proposals)

    print("\n🤖 Supervisor Final Decision:")
    print(supervisor_output)

    return {
        "decision": decision,
        "proposals": proposals,
        "supervisor": supervisor_output
    }

print("✅ Interactive decision engine ready.")


✅ Interactive decision engine ready.


In [109]:
result = run_interactive_decision()


🧠 AI Decision Assistant — Interactive Mode
------------------------------------------



Enter your decision question:
>  should i watch netflix or go to gym

Enter OPTION 1:
>  watch netflix

Enter OPTION 2:
>  go to gym



(Optional) Provide context (press Enter to skip):


Stress level (0–10):  5
Financial pressure (low/medium/high):  3
Energy level (0–10):  7



📌 Running decision through agents...

🔍 Agent Proposals:
{'agent_name': 'EmotionAgent', 'preferred_option': 'go to gym', 'scores': {'watch netflix': {'emotional_relief': 0.74, 'stress_risk': 0.37}, 'go to gym': {'emotional_relief': 0.8, 'stress_risk': 0.388}}, 'explanation': "I chose 'go to gym' because, given your stress level (5.0/10) and energy level (7.0/10), it offers the best emotional relief with the lowest stress risk among the options."} 

{'agent_name': 'LogicAgent', 'preferred_option': 'go to gym', 'scores': {'watch netflix': {'effort_required': 0.24, 'time_cost': 0.09, 'productivity_gain': 0.34, 'financial_score': 0.5, 'risk_factor': 0.0}, 'go to gym': {'effort_required': 0.26, 'time_cost': 0.11, 'productivity_gain': 0.94, 'financial_score': 0.5, 'risk_factor': 0.0}}, 'explanation': "I chose 'go to gym' because it has the best balance of logical factors: higher productivity, reasonable effort and time cost, lower risk, and financial suitability."} 

{'agent_name': 'LongTer

In [57]:
# STEP 1: Set up Gemini API key and import ADK components

import os
from kaggle_secrets import UserSecretsClient

# 1) Load your Gemini API key from Kaggle secrets
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key loaded from Kaggle secrets.")
except Exception as e:
    raise RuntimeError(
        "❌ Could not load GOOGLE_API_KEY from Kaggle secrets. "
        "Make sure you've added it in 'Settings' → 'Secrets'."
    ) from e

# 2) Import Gemini + ADK core components (same style as course notebooks)

from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK + Gemini imports successful.")

# 3) Create ADK Gemini model wrapper (required by Agent)

# Must use the exact model name supported by your API:
# From your list, this one exists:
# models/gemini-pro-latest

model = Gemini(model="models/gemini-2.5-flash")

print("✅ ADK Gemini model wrapper created.")



✅ Gemini API key loaded from Kaggle secrets.
✅ ADK + Gemini imports successful.
✅ ADK Gemini model wrapper created.


In [61]:
#STEP2 
placeholder_agent = Agent(
    name="placeholder_agent",
    model=model,
    description="A minimal placeholder agent required for initializing the runner.",
    tools=[]
)





In [59]:
#step 3: Inmemory Runner
runner = InMemoryRunner(
    agent=placeholder_agent,
    app_name="DecisionApp"
)

print("✅ InMemoryRunner initialized successfully.")


✅ InMemoryRunner initialized successfully.


In [62]:
# STEP 4: Define core data structures and helper functions (NO AI calls yet)

from typing import List, Dict, Any

# -------------------------------
# 1) Decision input format
# -------------------------------
def make_decision(question: str, options: List[str], context: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "question": question,
        "options": options,
        "context": context
    }


# -------------------------------
# 2) Standard Agent Output Schema
# -------------------------------
def make_agent_proposal(agent_name: str, preferred_option: str, scores: Dict[str, Any], explanation: str) -> Dict[str, Any]:
    return {
        "agent_name": agent_name,
        "preferred_option": preferred_option,
        "scores": scores,
        "explanation": explanation
    }


# -------------------------------
# 3) Helper: Get highest scoring option
# -------------------------------
def choose_best_option(score_dict: Dict[str, float]) -> str:
    return max(score_dict, key=score_dict.get)


# -------------------------------
# 4) Base Tool Interface (for future tools)
# -------------------------------
class BaseTool:
    def __init__(self, name):
        self.name = name
    
    def run(self, question: str, options: List[str], context: Dict[str, Any]) -> Dict[str, Any]:
        """
        Override this in child classes.
        Should return a dict with scores per option.
        """
        raise NotImplementedError("Each tool must implement run().")


print("✅ STEP 4 complete: Base structures & interfaces created.")


✅ STEP 4 complete: Base structures & interfaces created.


In [64]:
# STEP 5 ==== EmotionAgent 2.0 – stress + energy aware ====

from typing import Dict, Any
import math

class EmotionScoringTool:
    def run(self, question: str, options, context: Dict[str, Any]):
        """
        Returns per-option emotional_relief & stress_risk in [0, 1],
        based on stress_level, energy_level, and the wording of each option.
        """
        stress_level = float(context.get("stress_level", 5.0))   # 0–10
        energy_level = float(context.get("energy_level", 5.0))   # 0–10

        s = max(0.0, min(stress_level / 10.0, 1.0))   # normalize 0–1
        e = max(0.0, min(energy_level / 10.0, 1.0))   # normalize 0–1

        scores = {}

        for option in options:
            text = option.lower()

            # Base neutral state
            emotional_relief = 0.5
            stress_risk = 0.5

            # Heuristic flags
            is_rest = any(k in text for k in [
                "netflix", "movie", "sleep", "rest", "break", "relax", "chill"
            ])
            is_exercise = any(k in text for k in [
                "gym", "run", "jog", "walk", "exercise", "workout", "yoga"
            ])
            is_workish = any(k in text for k in [
                "study", "assignment", "project", "work", "email", "clean"
            ])

            # 1) High stress → rest looks extra soothing
            if is_rest:
                emotional_relief += 0.3 * (0.3 + s)     # more stressed → more relief
                stress_risk -= 0.2 * (0.4 + s / 2.0)   # rest reduces perceived stress

            # 2) Exercise is a special case: needs energy but relieves stress long term
            if is_exercise:
                if e >= 0.5:
                    # Enough energy → feels good
                    emotional_relief += 0.2 + 0.2 * s      # esp. if stressed
                    stress_risk -= 0.15 * (0.5 + s / 2.0)
                else:
                    # Too tired → feels like a burden
                    emotional_relief -= 0.15 * (0.5 + s)
                    stress_risk += 0.25 * (0.5 + s)

            # 3) Work-ish options: can increase short-term stress, esp. if already stressed
            if is_workish:
                if s >= 0.6:
                    # Already stressed → this feels heavy
                    emotional_relief -= 0.2 * (0.5 + s)
                    stress_risk += 0.25 * (0.5 + s)
                else:
                    # Low stress → can feel productive & slightly relieving
                    emotional_relief += 0.1 * (0.5 - s)
                    stress_risk += 0.05 * s

            # 4) If energy is extremely low, any effortful thing feels bad
            if e <= 0.3 and (is_exercise or is_workish):
                emotional_relief -= 0.1 * (0.4 + (0.3 - e))
                stress_risk += 0.15 * (0.4 + (0.3 - e))

            # Clamp to [0, 1]
            emotional_relief = max(0.0, min(emotional_relief, 1.0))
            stress_risk = max(0.0, min(stress_risk, 1.0))

            scores[option] = {
                "emotional_relief": round(emotional_relief, 3),
                "stress_risk": round(stress_risk, 3),
            }

        return scores


class EmotionAgent:
    def __init__(self, tool: EmotionScoringTool):
        self.tool = tool

    def decide(self, decision: Dict[str, Any]) -> Dict[str, Any]:
        question = decision["question"]
        options = decision["options"]
        context = decision.get("context", {})

        scores = self.tool.run(question, options, context)

        # Combine into a single emotional score: relief - stress
        best_option = None
        best_score = -999

        for option in options:
            sc = scores[option]
            combined = sc["emotional_relief"] - sc["stress_risk"]
            if combined > best_score:
                best_score = combined
                best_option = option

        # Build a simple natural-language explanation
        stress_level = context.get("stress_level", "unknown")
        energy_level = context.get("energy_level", "unknown")

        explanation = (
            f"I chose '{best_option}' because, given your stress level "
            f"({stress_level}/10) and energy level ({energy_level}/10), it offers "
            f"the best emotional relief with the lowest stress risk among the options."
        )

        return {
            "agent_name": "EmotionAgent",
            "preferred_option": best_option,
            "scores": scores,
            "explanation": explanation,
        }


# Recreate the global emotion_agent instance
emotion_agent = EmotionAgent(EmotionScoringTool())

print("✅ EmotionAgent 2.0 loaded (stress + energy aware).")


✅ EmotionAgent 2.0 loaded (stress + energy aware).


In [69]:
# step 6: ------------ UPGRADED INPUT-INDEPENDENT LOGIC AGENT (Context-Aware) ------------

import re
import numpy as np

class LogicScoringTool:
    def run(self, question, options, context):
        stress = context.get("stress_level", 5)
        energy = context.get("energy_level", 5)
        money_pressure = context.get("financial_pressure", "medium")

        results = {}

        # Normalize helper
        def norm(x):
            return round(max(0.0, min(1.0, x)), 2)

        # Financial pressure multiplier (affects general financial reasoning)
        fin_mult = {"low": 0.5, "medium": 1.0, "high": 1.3}.get(money_pressure, 1.0)

        for opt in options:
            text = opt.lower()

            # Count linguistic features
            verbs = len(re.findall(r"\b\w+(?=ing\b|\bgo\b|\bdo\b|\bmake\b|\bwork\b)\b", text))
            nouns = len(re.findall(r"\b[a-zA-Z]{4,}\b", text))
            words = len(text.split())

            # Detect uncertainty words
            uncertain = len(re.findall(r"\b(maybe|might|could|possibly|unsure|uncertain)\b", text))

            # 1. Effort score (longer + more verbs → higher effort)
            effort = 0.3 * (verbs / (words + 1)) + 0.3 * (words / 15) + 0.4 * (stress / 10)

            # 2. Time cost (more words/subclauses → longer time)
            subclauses = len(re.findall(r"(because|and|then|after|while|but)", text))
            time_cost = 0.4 * (words / 20) + 0.4 * (subclauses / 3) + 0.2 * (effort)

            # 3. Productivity (imperative verbs → higher)
            imperative = len(re.findall(r"^(start|go|do|make|create|plan|organize)", text))
            passive = len(re.findall(r"\b(be|stay|remain|wait)\b", text))

            productivity = (
                0.6 * (imperative / (1 + verbs)) +
                0.2 * (1 - passive / (1 + words)) +
                0.2 * (energy / 10)
            )

            # 4. Financial impact (context dependent)
            financial_impact = norm(0.5 * fin_mult)

            # 5. Risk score (uncertainty words)
            risk = norm(uncertain / 3)

            results[opt] = {
                "effort_required": norm(effort),
                "time_cost": norm(time_cost),
                "productivity_gain": norm(productivity),
                "financial_score": financial_impact,
                "risk_factor": risk
            }

        return results

class LogicAgent:
    def __init__(self, tool):
        self.tool = tool

    def decide(self, decision):
        question = decision["question"]
        options = decision["options"]
        context = decision.get("context", {})

        scores = self.tool.run(question, options, context)

        best_opt = None
        best_score = -1e9

        for opt, s in scores.items():
            # Combined logic score — input-independent
            # Higher productivity, lower effort, lower risk, lower time cost is better.

            combined = (
                1.0 * s["productivity_gain"] +
                0.6 * (1 - s["time_cost"]) +
                0.5 * (1 - s["effort_required"]) +
                0.4 * s["financial_score"] +
                0.3 * (1 - s["risk_factor"])
            )

            if combined > best_score:
                best_score = combined
                best_opt = opt

        explanation = (
            f"I chose '{best_opt}' because it has the best balance of logical factors: "
            f"higher productivity, reasonable effort and time cost, lower risk, and financial suitability."
        )

        return {
            "agent_name": "LogicAgent",
            "preferred_option": best_opt,
            "scores": scores,
            "explanation": explanation
        }


In [70]:
# ========= STEP 7: UPGRADED LONG-TERM AGENT (context-aware, input-independent) =========

import re
from typing import Dict, Any

class LongTermScoringTool:
    def run(self, question: str, options, context: Dict[str, Any]):
        """
        Compute long-term gain and regret for each option using:
        - User context: goal_importance, risk_tolerance, stability_preference
        - Generic text features: length, future vs immediate wording, uncertainty
        No domain-specific keywords (gym/study/etc).
        """

        # User context (all optional, with safe defaults)
        goal_importance = float(context.get("goal_importance", 7.0))      # 0–10
        risk_tolerance = context.get("risk_tolerance", "medium")          # "low"/"medium"/"high"
        stability_pref = context.get("stability_preference", "medium")    # "low"/"medium"/"high"

        # Normalize helpers
        def clamp01(x: float) -> float:
            return max(0.0, min(1.0, x))

        goal_norm = clamp01(goal_importance / 10.0)

        risk_map = {"low": 0.3, "medium": 0.5, "high": 0.7}
        stability_map = {"low": 0.3, "medium": 0.5, "high": 0.7}

        risk_t = risk_map.get(risk_tolerance, 0.5)
        stab_t = stability_map.get(stability_pref, 0.5)

        scores: Dict[str, Dict[str, float]] = {}

        for opt in options:
            text = opt.lower().strip()
            words = text.split()
            n_words = len(words)

            # Generic linguistic features (no domain-specific tokens)
            future_markers = len(re.findall(
                r"\b(later|future|long term|long-term|years|next year|next month|eventually)\b",
                text
            ))
            immediate_markers = len(re.findall(
                r"\b(now|today|tonight|this week|right away|immediately|just for today)\b",
                text
            ))
            uncertainty_markers = len(re.findall(
                r"\b(maybe|might|could|possibly|unsure|not sure|consider)\b",
                text
            ))

            # 1) Long-term gain:
            # - higher when:
            #   * goal importance is high
            #   * more future-oriented wording
            #   * options look a bit "committing" (longer / more detailed)
            length_factor = clamp01(n_words / 20.0)  # 0 for very short, up to ~1 for long
            future_bias = clamp01(future_markers / (1.0 + immediate_markers))

            future_gain = (
                0.5 * goal_norm +          # how important this decision is overall
                0.3 * future_bias +        # how future-oriented the option sounds
                0.2 * length_factor        # longer/more structured → more like a plan
            )

            # 2) Future regret:
            # - higher when:
            #   * goal is important but gain is low
            #   * uncertainty wording is high
            #   * user has low risk tolerance
            regret_from_goal = goal_norm * (1.0 - future_gain)       # big goal, weak gain → regret
            regret_from_uncertainty = clamp01(uncertainty_markers / 3.0)
            regret_from_risk = (1.0 - risk_t)                        # low risk tolerance → more regret

            future_regret = (
                0.5 * regret_from_goal +
                0.3 * regret_from_uncertainty +
                0.2 * regret_from_risk
            )

            future_gain = clamp01(future_gain)
            future_regret = clamp01(future_regret)

            scores[opt] = {
                "future_gain": round(future_gain, 3),
                "future_regret": round(future_regret, 3),
            }

        return scores

class LongTermAgent:
    def __init__(self, tool: LongTermScoringTool):
        self.tool = tool

    def decide(self, decision: Dict[str, Any]) -> Dict[str, Any]:
        question = decision["question"]
        options = decision["options"]
        context = decision.get("context", {})

        scores = self.tool.run(question, options, context)

        best_option = None
        best_score = -1e9

        for opt in options:
            sc = scores[opt]
            # Combined long-term score: high gain, low regret
            combined = sc["future_gain"] - sc["future_regret"]
            if combined > best_score:
                best_score = combined
                best_option = opt

        goal_importance = context.get("goal_importance", "unknown")
        risk_tolerance = context.get("risk_tolerance", "medium")

        explanation = (
            f"I chose '{best_option}' because, over the long run, it offers the best "
            f"trade-off between future gain and future regret given your goal importance "
            f"({goal_importance}/10) and risk tolerance ('{risk_tolerance}')."
        )

        return {
            "agent_name": "LongTermAgent",
            "preferred_option": best_option,
            "scores": scores,
            "explanation": explanation,
        }
print("✅ LongTermAgent upgraded: future_gain / future_regret are now context-aware & input-independent.")

✅ LongTermAgent upgraded: future_gain / future_regret are now context-aware & input-independent.


In [71]:
# ================== STEP 8: UPGRADED INPUT-INDEPENDENT VALUES AGENT ==================

import re
from typing import Dict, Any

class ValuesScoringTool:
    def run(self, question: str, options, context: Dict[str, Any]):

        # User-stated values (0–10 each), with safe defaults
        values = context.get("values", {})
        career_w = values.get("career", 5) / 10
        relation_w = values.get("relationships", 5) / 10
        health_w = values.get("health", 5) / 10
        finance_w = values.get("finance", 5) / 10
        peace_w = values.get("peace", 5) / 10

        def clamp01(x):
            return max(0.0, min(1.0, x))

        scores = {}

        for opt in options:
            text = opt.lower().strip()
            words = text.split()
            n_words = len(words)

            # Linguistic indicators
            assertive = len(re.findall(r"\b(start|build|commit|create|improve|develop)\b", text))
            calming = len(re.findall(r"\b(rest|pause|breathe|reflect|break)\b", text))
            social = len(re.findall(r"\b(friend|family|partner|talk|share|connect)\b", text))
            money_terms = len(re.findall(r"\b(budget|money|salary|cost|invest|pay|save)\b", text))

            future_terms = len(re.findall(r"\b(future|long term|career|growth|progress|develop)\b", text))
            uncertainty = len(re.findall(r"\b(maybe|might|could|unsure|possibly|not sure)\b", text))

            # 1. Career alignment
            career_align = clamp01(
                0.6 * (assertive / (1 + n_words)) +
                0.4 * (future_terms / (1 + n_words))
            ) * career_w

            # 2. Relationship alignment
            relation_align = clamp01(
                social / (1 + n_words)
            ) * relation_w

            # 3. Health alignment (mental + physical)
            health_align = clamp01(
                calming / (1 + uncertainty + n_words)
            ) * health_w

            # 4. Financial alignment
            financial_align = clamp01(
                money_terms / (1 + n_words)
            ) * finance_w

            # 5. Inner peace / stress reduction alignment
            peace_align = clamp01(
                calming / (1 + uncertainty) +
                0.2 * (1 - (n_words / 20))
            ) * peace_w

            # Weighted total
            weighted = round(
                career_align +
                relation_align +
                health_align +
                financial_align +
                peace_align,
                3
            )

            scores[opt] = {
                "career_alignment": round(career_align, 3),
                "relationship_alignment": round(relation_align, 3),
                "health_alignment": round(health_align, 3),
                "financial_alignment": round(financial_align, 3),
                "peace_alignment": round(peace_align, 3),
                "weighted_score": weighted
            }

        return scores

class ValuesAgent:
    def __init__(self, tool):
        self.tool = tool

    def decide(self, decision: Dict[str, Any]):
        question = decision["question"]
        options = decision["options"]
        context = decision.get("context", {})

        scores = self.tool.run(question, options, context)

        best_option = None
        best_score = -1e9

        for opt in options:
            if scores[opt]["weighted_score"] > best_score:
                best_option = opt
                best_score = scores[opt]["weighted_score"]

        explanation = (
            f"I chose '{best_option}' because it best aligns with your stated values "
            f"(career={context.get('values', {}).get('career', 5)}, "
            f"relationships={context.get('values', {}).get('relationships', 5)}, "
            f"health={context.get('values', {}).get('health', 5)}, "
            f"finance={context.get('values', {}).get('finance', 5)}, "
            f"peace={context.get('values', {}).get('peace', 5)})."
        )

        return {
            "agent_name": "ValuesAgent",
            "preferred_option": best_option,
            "scores": scores,
            "explanation": explanation
        }


In [72]:
values_agent = ValuesAgent(ValuesScoringTool())
print("✅ ValuesAgent upgraded successfully — deeply personalized and input-independent.")


✅ ValuesAgent upgraded successfully — deeply personalized and input-independent.


In [73]:
# STEP 9 — Create all upgraded agents

emotion_tool = EmotionScoringTool()
logic_tool = LogicScoringTool()
longterm_tool = LongTermScoringTool()
values_tool = ValuesScoringTool()

emotion_agent = EmotionAgent(emotion_tool)
logic_agent = LogicAgent(logic_tool)
longterm_agent = LongTermAgent(longterm_tool)
values_agent = ValuesAgent(values_tool)

print("✅ All upgraded agents created successfully!")


✅ All upgraded agents created successfully!


In [77]:
#step 10: Sample decision maker test
sample_decision = make_decision(
    question="Should I work late tonight or go to dinner with friends?",
    options=["Work late", "Go to dinner"],
    context={
        "stress_level": 8,
        "recent_social_time": "very_low",
        "deadline_tomorrow": True,
        "price": 40,
        "savings_balance": 500,
        "physical_fatigue": 6,
        "sleep_hours_last_night": 5,
        "workout_consistency": "medium",
        "energy_level": 6,
        "financial_pressure": "medium",
        "user_values": {
            "health": 0.7,
            "relationships": 0.9,
            "career": 0.6,
            "financial_stability": 0.5,
            "peace": 0.8
        }
    }
)

emotion_result = emotion_agent.decide(sample_decision)
logic_result = logic_agent.decide(sample_decision)
longterm_result = longterm_agent.decide(sample_decision)
values_result = values_agent.decide(sample_decision)

print("🔍 EmotionAgent:\n", emotion_result, "\n")
print("🔍 LogicAgent:\n", logic_result, "\n")
print("🔍 LongTermAgent:\n", longterm_result, "\n")
print("🔍 ValuesAgent:\n", values_result, "\n")


🔍 EmotionAgent:
 {'agent_name': 'EmotionAgent', 'preferred_option': 'Go to dinner', 'scores': {'Work late': {'emotional_relief': 0.24, 'stress_risk': 0.825}, 'Go to dinner': {'emotional_relief': 0.5, 'stress_risk': 0.5}}, 'explanation': "I chose 'Go to dinner' because, given your stress level (8/10) and energy level (6/10), it offers the best emotional relief with the lowest stress risk among the options."} 

🔍 LogicAgent:
 {'agent_name': 'LogicAgent', 'preferred_option': 'Go to dinner', 'scores': {'Work late': {'effort_required': 0.36, 'time_cost': 0.11, 'productivity_gain': 0.32, 'financial_score': 0.5, 'risk_factor': 0.0}, 'Go to dinner': {'effort_required': 0.38, 'time_cost': 0.14, 'productivity_gain': 0.92, 'financial_score': 0.5, 'risk_factor': 0.0}}, 'explanation': "I chose 'Go to dinner' because it has the best balance of logical factors: higher productivity, reasonable effort and time cost, lower risk, and financial suitability."} 

🔍 LongTermAgent:
 {'agent_name': 'LongTermAg

In [79]:
# STEP 11: Create the LLM-powered Supervisor Agent (ADK Agent)

import json

supervisor_agent = Agent(
    name="SupervisorAgent",
    model=model,  # uses the Gemini model you initialized earlier
    description="An LLM supervisor that reads all agent proposals and makes the final decision.",
    tools=[]  # no external tools needed here; LLM only
)

print("✅ STEP 11 complete: SupervisorAgent created.")


✅ STEP 11 complete: SupervisorAgent created.


In [80]:
# step 12

def run_supervisor_llm(proposals):
    prompt = f"""
You are the SupervisorAgent in a multi-agent decision negotiation system.

You receive decision proposals from 4 agents:
- EmotionAgent
- LogicAgent
- LongTermAgent
- ValuesAgent

Your job:
1. Analyze all proposals.
2. Resolve any disagreements.
3. Choose the BEST final decision.
4. Return STRICT JSON ONLY in the following structure:

{{
  "final_decision": "...",
  "reasoning": "...",
  "agent_votes": {{
    "EmotionAgent": "...",
    "LogicAgent": "...",
    "LongTermAgent": "...",
    "ValuesAgent": "..."
  }}
}}
"""

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",   # Works perfectly in Kaggle
        contents=[
            prompt,
            "Here are the agent proposals:",
            json.dumps(proposals, indent=2)
        ]
    )

    return response.text


In [81]:
# STEP 13 — Gather proposals from all internal agents

proposals = [
    emotion_agent.decide(sample_decision),
    logic_agent.decide(sample_decision),
    longterm_agent.decide(sample_decision),
    values_agent.decide(sample_decision)
]

print("Collected proposals:")
import json
print(json.dumps(proposals, indent=2))


Collected proposals:
[
  {
    "agent_name": "EmotionAgent",
    "preferred_option": "Go to dinner",
    "scores": {
      "Work late": {
        "emotional_relief": 0.24,
        "stress_risk": 0.825
      },
      "Go to dinner": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      }
    },
    "explanation": "I chose 'Go to dinner' because, given your stress level (8/10) and energy level (6/10), it offers the best emotional relief with the lowest stress risk among the options."
  },
  {
    "agent_name": "LogicAgent",
    "preferred_option": "Go to dinner",
    "scores": {
      "Work late": {
        "effort_required": 0.36,
        "time_cost": 0.11,
        "productivity_gain": 0.32,
        "financial_score": 0.5,
        "risk_factor": 0.0
      },
      "Go to dinner": {
        "effort_required": 0.38,
        "time_cost": 0.14,
        "productivity_gain": 0.92,
        "financial_score": 0.5,
        "risk_factor": 0.0
      }
    },
    "explanation": "I cho

In [84]:
#step 14 
from google.genai import Client

client = Client(api_key=GOOGLE_API_KEY)

print("Client ready.")



Client ready.


In [85]:
#step 15
supervisor_output = run_supervisor_llm(proposals)

print("SUPERVISOR OUTPUT:")
print(supervisor_output)


SUPERVISOR OUTPUT:
```json
{
  "final_decision": "Go to dinner",
  "reasoning": "Three out of four agents (EmotionAgent, LogicAgent, LongTermAgent) strongly recommend 'Go to dinner'. Their reasoning collectively covers immediate emotional well-being (reduced stress), logical efficiency (productivity gain, reasonable effort/cost), and long-term benefit (better future gain/regret balance). The ValuesAgent, while preferring 'Work late', does so based on an extremely marginal difference in 'peace_alignment' (0.09 vs 0.085) and zero alignment across all other stated values for both options, making its dissenting vote less compelling. Therefore, 'Go to dinner' represents the most balanced and well-supported decision across immediate, logical, and long-term perspectives.",
  "agent_votes": {
    "EmotionAgent": "Go to dinner",
    "LogicAgent": "Go to dinner",
    "LongTermAgent": "Go to dinner",
    "ValuesAgent": "Work late"
  }
}
```


In [86]:
# STEP 16 

# FINAL EVALUATION FUNCTION — handles code fences + always returns a result dict

import json

def evaluate_test_case(test_name, user_decision):
    print(f"\n📌 Running Test Case: {test_name}")
    print(f"User Decision: {user_decision['question']}\n")

    # 1. Generate proposals
    proposals = [
        emotion_agent.decide(user_decision),
        logic_agent.decide(user_decision),
        longterm_agent.decide(user_decision),
        values_agent.decide(user_decision)
    ]

    print("🔍 Internal Agent Proposals:")
    print(json.dumps(proposals, indent=2))

    # 2. Run supervisor
    supervisor_output = run_supervisor_llm(proposals)

    print("\n🤖 Supervisor Output (Raw):")
    print(supervisor_output)

    # 3. Clean output for JSON parsing
    cleaned_output = supervisor_output.strip()

    # Remove markdown fences: ```json ... ```
    if cleaned_output.startswith("```"):
        cleaned_output = cleaned_output.strip("`")   # remove backticks
        cleaned_output = cleaned_output.replace("json", "", 1).strip()

    # 4. Parse JSON safely
    try:
        supervisor_json = json.loads(cleaned_output)
        print("\n✅ Parsed Supervisor JSON:")
        print(json.dumps(supervisor_json, indent=2))

        # SUCCESS RESULT
        return {
            "test": test_name,
            "result": "OK",
            "supervisor": supervisor_json
        }

    except Exception as e:
        print("\n❌ FAILED: Supervisor did not return valid JSON.")
        print("Error:", e)
        print("\nCleaned Output:\n", cleaned_output)

        # FAIL RESULT
        return {
            "test": test_name,
            "result": "FAIL",
            "reason": "Invalid JSON"
        }

print("✅ Evaluation helper function updated & finalized.")


✅ Evaluation helper function updated & finalized.


In [87]:

# STEP 16.2 — Test Case 1: Ambiguous Decision (with proper context)

ambiguous_decision = {
    "question": "Should I study or clean my room?",
    "options": ["Study", "Clean room"],
    "context": {}   # IMPORTANT: must be a dict
}

result_ambiguous = evaluate_test_case("Ambiguous Decision Test", ambiguous_decision)

print("\n📊 TEST RESULT:", result_ambiguous["result"])





📌 Running Test Case: Ambiguous Decision Test
User Decision: Should I study or clean my room?

🔍 Internal Agent Proposals:
[
  {
    "agent_name": "EmotionAgent",
    "preferred_option": "Study",
    "scores": {
      "Study": {
        "emotional_relief": 0.5,
        "stress_risk": 0.525
      },
      "Clean room": {
        "emotional_relief": 0.5,
        "stress_risk": 0.525
      }
    },
    "explanation": "I chose 'Study' because, given your stress level (unknown/10) and energy level (unknown/10), it offers the best emotional relief with the lowest stress risk among the options."
  },
  {
    "agent_name": "LogicAgent",
    "preferred_option": "Study",
    "scores": {
      "Study": {
        "effort_required": 0.22,
        "time_cost": 0.06,
        "productivity_gain": 0.3,
        "financial_score": 0.5,
        "risk_factor": 0.0
      },
      "Clean room": {
        "effort_required": 0.24,
        "time_cost": 0.09,
        "productivity_gain": 0.3,
        "financial_

In [88]:
# STEP 17 — Test Case 2: Invalid Options

invalid_decision = {
    "question": "Should I travel to the moon tonight or bake a pizza?",
    "options": ["Travel to the moon tonight", "Bake a pizza"],
    "context": {}
}

result_invalid = evaluate_test_case("Invalid Options Test", invalid_decision)

print("\n📊 TEST RESULT:", result_invalid["result"])



📌 Running Test Case: Invalid Options Test
User Decision: Should I travel to the moon tonight or bake a pizza?

🔍 Internal Agent Proposals:
[
  {
    "agent_name": "EmotionAgent",
    "preferred_option": "Travel to the moon tonight",
    "scores": {
      "Travel to the moon tonight": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      },
      "Bake a pizza": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      }
    },
    "explanation": "I chose 'Travel to the moon tonight' because, given your stress level (unknown/10) and energy level (unknown/10), it offers the best emotional relief with the lowest stress risk among the options."
  },
  {
    "agent_name": "LogicAgent",
    "preferred_option": "Bake a pizza",
    "scores": {
      "Travel to the moon tonight": {
        "effort_required": 0.3,
        "time_cost": 0.16,
        "productivity_gain": 0.3,
        "financial_score": 0.5,
        "risk_factor": 0.0
      },
      "Bake a pizza": {
      

In [89]:
# STEP 18 — Test Case 3: Complex Multi-Factor Decision

complex_decision = {
    "question": "Should I accept the promotion that doubles my work hours but increases salary, "
                "or stay in my current role with less pay but more flexibility?",
    "options": [
        "Accept the promotion with doubled hours and higher salary",
        "Stay in current role with less pay and more flexibility"
    ],
    "context": {
        "stress_level": 6,
        "career_pressure": "high",
        "health_state": "medium",
        "recent_social_time": "low",
        "value_weights": {
            "career": 0.9,
            "health": 0.7,
            "relationships": 0.8,
            "financial": 0.9,
            "peace": 0.8
        }
    }
}

result_complex = evaluate_test_case("Complex Multi-Factor Decision Test", complex_decision)

print("\n📊 TEST RESULT:", result_complex["result"])



📌 Running Test Case: Complex Multi-Factor Decision Test
User Decision: Should I accept the promotion that doubles my work hours but increases salary, or stay in my current role with less pay but more flexibility?

🔍 Internal Agent Proposals:
[
  {
    "agent_name": "EmotionAgent",
    "preferred_option": "Accept the promotion with doubled hours and higher salary",
    "scores": {
      "Accept the promotion with doubled hours and higher salary": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      },
      "Stay in current role with less pay and more flexibility": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      }
    },
    "explanation": "I chose 'Accept the promotion with doubled hours and higher salary' because, given your stress level (6/10) and energy level (unknown/10), it offers the best emotional relief with the lowest stress risk among the options."
  },
  {
    "agent_name": "LogicAgent",
    "preferred_option": "Accept the promotion with d

In [90]:
# STEP 18 — Test Case 3: Complex Multi-Factor Decision

complex_decision = {
    "question": "Should I accept the promotion that doubles my work hours but increases salary, "
                "or stay in my current role with less pay but more flexibility?",
    "options": [
        "Accept the promotion with doubled hours and higher salary",
        "Stay in current role with less pay and more flexibility"
    ],
    "context": {
        "stress_level": 6,
        "career_pressure": "high",
        "health_state": "medium",
        "recent_social_time": "low",
        "value_weights": {
            "career": 0.9,
            "health": 0.7,
            "relationships": 0.8,
            "financial": 0.9,
            "peace": 0.8
        }
    }
}

result_complex = evaluate_test_case("Complex Multi-Factor Decision Test", complex_decision)

print("\n📊 TEST RESULT:", result_complex["result"])



📌 Running Test Case: Complex Multi-Factor Decision Test
User Decision: Should I accept the promotion that doubles my work hours but increases salary, or stay in my current role with less pay but more flexibility?

🔍 Internal Agent Proposals:
[
  {
    "agent_name": "EmotionAgent",
    "preferred_option": "Accept the promotion with doubled hours and higher salary",
    "scores": {
      "Accept the promotion with doubled hours and higher salary": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      },
      "Stay in current role with less pay and more flexibility": {
        "emotional_relief": 0.5,
        "stress_risk": 0.5
      }
    },
    "explanation": "I chose 'Accept the promotion with doubled hours and higher salary' because, given your stress level (6/10) and energy level (unknown/10), it offers the best emotional relief with the lowest stress risk among the options."
  },
  {
    "agent_name": "LogicAgent",
    "preferred_option": "Accept the promotion with d

In [91]:
# STEP 19 — Observability: run a full decision with a structured trace

def run_decision_with_trace(test_name, decision):
    print(f"\n🔎 TRACE RUN: {test_name}")
    print(f"Question: {decision['question']}")
    print(f"Options: {decision['options']}\n")

    # 1) Collect proposals from all agents
    proposals = [
        emotion_agent.decide(decision),
        logic_agent.decide(decision),
        longterm_agent.decide(decision),
        values_agent.decide(decision),
    ]

    # Compact summary: who prefers what?
    print("🧩 Agent votes:")
    for p in proposals:
        print(f"  - {p['agent_name']}: {p['preferred_option']}")

    # 2) Call supervisor
    supervisor_output = run_supervisor_llm(proposals)

    print("\n🧠 Supervisor raw output:")
    print(supervisor_output)

    # 3) Reuse the same JSON cleaning logic
    cleaned_output = supervisor_output.strip()
    if cleaned_output.startswith("```"):
        cleaned_output = cleaned_output.strip("`")
        cleaned_output = cleaned_output.replace("json", "", 1).strip()

    try:
        supervisor_json = json.loads(cleaned_output)
        print("\n✅ Parsed Supervisor JSON (summary):")
        print(f"  Final decision: {supervisor_json.get('final_decision')}")
        print(f"  Reason (truncated): {supervisor_json.get('reasoning', '')[:200]}...")
    except Exception as e:
        print("\n❌ Could not parse supervisor JSON in trace mode.")
        print("Error:", e)
        supervisor_json = None

    # 4) Return a structured trace object for later analysis / logging
    return {
        "test": test_name,
        "question": decision["question"],
        "options": decision["options"],
        "proposals": proposals,
        "supervisor_raw": supervisor_output,
        "supervisor": supervisor_json,
    }

print("✅ STEP 19.1: run_decision_with_trace() ready.")


✅ STEP 19.1: run_decision_with_trace() ready.


In [92]:
trace_output = run_decision_with_trace(
    "Observability Test — Promotion Scenario",
    complex_decision
)

trace_output



🔎 TRACE RUN: Observability Test — Promotion Scenario
Question: Should I accept the promotion that doubles my work hours but increases salary, or stay in my current role with less pay but more flexibility?
Options: ['Accept the promotion with doubled hours and higher salary', 'Stay in current role with less pay and more flexibility']

🧩 Agent votes:
  - EmotionAgent: Accept the promotion with doubled hours and higher salary
  - LogicAgent: Accept the promotion with doubled hours and higher salary
  - LongTermAgent: Stay in current role with less pay and more flexibility
  - ValuesAgent: Accept the promotion with doubled hours and higher salary

🧠 Supervisor raw output:
```json
{
  "final_decision": "Accept the promotion with doubled hours and higher salary",
  "reasoning": "The majority of agents (EmotionAgent, LogicAgent, and ValuesAgent) propose 'Accept the promotion with doubled hours and higher salary'. EmotionAgent indicates this option provides the best emotional relief and lowes

{'test': 'Observability Test — Promotion Scenario',
 'question': 'Should I accept the promotion that doubles my work hours but increases salary, or stay in my current role with less pay but more flexibility?',
 'options': ['Accept the promotion with doubled hours and higher salary',
  'Stay in current role with less pay and more flexibility'],
 'proposals': [{'agent_name': 'EmotionAgent',
   'preferred_option': 'Accept the promotion with doubled hours and higher salary',
   'scores': {'Accept the promotion with doubled hours and higher salary': {'emotional_relief': 0.5,
     'stress_risk': 0.5},
    'Stay in current role with less pay and more flexibility': {'emotional_relief': 0.5,
     'stress_risk': 0.5}},
   'explanation': "I chose 'Accept the promotion with doubled hours and higher salary' because, given your stress level (6/10) and energy level (unknown/10), it offers the best emotional relief with the lowest stress risk among the options."},
  {'agent_name': 'LogicAgent',
   'pr

In [93]:
# STEP 19.3 — Pretty Visualization Helpers

from tabulate import tabulate

def visualize_trace(trace):
    proposals = trace["proposals"]
    options = trace["options"]

    print("\n========================")
    print("🧩 AGENT VOTE SUMMARY")
    print("========================")

    vote_table = []
    for p in proposals:
        vote_table.append([p["agent_name"], p["preferred_option"]])
    print(tabulate(vote_table, headers=["Agent", "Preferred Option"], tablefmt="fancy_grid"))

    print("\n========================")
    print("📊 SCORE MATRIX")
    print("========================")

    # Build matrix rows: agent × option with a weighted score if available
    matrix_rows = []
    for p in proposals:
        row = [p["agent_name"]]
        for opt in options:
            scores = p["scores"].get(opt, {})
            # compute a simple average to display
            if len(scores) > 0:
                avg_score = sum(scores.values()) / len(scores)
                row.append(f"{avg_score:.3f}")
            else:
                row.append("-")
        matrix_rows.append(row)

    print(tabulate(matrix_rows, headers=["Agent"] + options, tablefmt="fancy_grid"))

    print("\n========================")
    print("🤖 SUPERVISOR DECISION")
    print("========================")
    supervisor = trace["supervisor"]
    if supervisor:
        print("Decision:", supervisor.get("final_decision"))
        print("Reason:", supervisor.get("reasoning")[:250] + "...")
    else:
        print("Supervisor JSON missing.")

    print("\n✨ Visualization complete.\n")

print("✅ STEP 19.3 visualization helpers installed.")


✅ STEP 19.3 visualization helpers installed.


In [94]:
visualize_trace(trace_output)



🧩 AGENT VOTE SUMMARY
╒═══════════════╤═══════════════════════════════════════════════════════════╕
│ Agent         │ Preferred Option                                          │
╞═══════════════╪═══════════════════════════════════════════════════════════╡
│ EmotionAgent  │ Accept the promotion with doubled hours and higher salary │
├───────────────┼───────────────────────────────────────────────────────────┤
│ LogicAgent    │ Accept the promotion with doubled hours and higher salary │
├───────────────┼───────────────────────────────────────────────────────────┤
│ LongTermAgent │ Stay in current role with less pay and more flexibility   │
├───────────────┼───────────────────────────────────────────────────────────┤
│ ValuesAgent   │ Accept the promotion with doubled hours and higher salary │
╘═══════════════╧═══════════════════════════════════════════════════════════╛

📊 SCORE MATRIX
╒═══════════════╤═════════════════════════════════════════════════════════════╤═════════════════════════

In [95]:
# FEATURE 1 — Multi-Agent Conflict Analyzer

def analyze_conflicts(proposals):
    """
    Computes disagreement between agents.
    Returns:
      - conflict_pairs: list of tuples (agentA, agentB, conflict_level)
      - overall_conflict_score: 0–1
    """

    # Extract preferred options
    prefs = {p["agent_name"]: p["preferred_option"] for p in proposals}

    agents = list(prefs.keys())
    conflict_pairs = []
    disagreements = 0
    total_pairs = 0

    # Pairwise comparison
    for i in range(len(agents)):
        for j in range(i+1, len(agents)):
            total_pairs += 1
            a1, a2 = agents[i], agents[j]
            o1, o2 = prefs[a1], prefs[a2]

            conflict = 1 if o1 != o2 else 0
            if conflict == 1:
                conflict_pairs.append((a1, a2, "DISAGREE"))
                disagreements += 1
            else:
                conflict_pairs.append((a1, a2, "AGREE"))

    # conflict percentage
    overall_conflict_score = disagreements / total_pairs if total_pairs > 0 else 0.0

    return {
        "pairs": conflict_pairs,
        "overall_conflict_score": round(overall_conflict_score, 3)
    }


print("✅ Conflict Analyzer installed.")


✅ Conflict Analyzer installed.


In [96]:
def display_conflict_analysis(conflict_data):
    print("\n==========================")
    print("⚔️  AGENT CONFLICT ANALYSIS")
    print("==========================")

    for a1, a2, status in conflict_data["pairs"]:
        if status == "DISAGREE":
            print(f"❌ {a1} vs {a2} → DISAGREE")
        else:
            print(f"✔️  {a1} vs {a2} → AGREE")

    print("\nOverall Conflict Score:", conflict_data["overall_conflict_score"])

    if conflict_data["overall_conflict_score"] == 0:
        print("🟢 Perfect Agreement (0 conflict)")
    elif conflict_data["overall_conflict_score"] < 0.34:
        print("🟡 Mild Conflict")
    elif conflict_data["overall_conflict_score"] < 0.67:
        print("🟠 Medium Conflict")
    else:
        print("🔴 HIGH Conflict — Supervisor Hard Case")
        
    print()


In [97]:
conflict_data = analyze_conflicts(trace_output["proposals"])
display_conflict_analysis(conflict_data)



⚔️  AGENT CONFLICT ANALYSIS
✔️  EmotionAgent vs LogicAgent → AGREE
❌ EmotionAgent vs LongTermAgent → DISAGREE
✔️  EmotionAgent vs ValuesAgent → AGREE
❌ LogicAgent vs LongTermAgent → DISAGREE
✔️  LogicAgent vs ValuesAgent → AGREE
❌ LongTermAgent vs ValuesAgent → DISAGREE

Overall Conflict Score: 0.5
🟠 Medium Conflict



In [98]:
# FEATURE 2 — Decision Stability Score (0–100)

import numpy as np

def compute_stability_score(proposals, supervisor_json=None):
    """
    Computes a stability score (0-100) combining:
      - agent consensus
      - score variance
      - value alignment consistency
      - supervisor-agent alignment
    """

    # 1) Agent consensus
    prefs = [p["preferred_option"] for p in proposals]
    most_common = max(set(prefs), key=prefs.count)
    consensus_ratio = prefs.count(most_common) / len(prefs)  # 0-1

    # 2) Score variance (averaged across agents and options)
    all_scores = []
    for p in proposals:
        for opt_scores in p["scores"].values():
            all_scores.extend(opt_scores.values())
    score_variance = np.var(all_scores) if len(all_scores) > 0 else 0
    # invert variance so higher variance → lower stability
    variance_score = max(0, 1 - min(score_variance, 1))

    # 3) Supervisor alignment
    if supervisor_json:
        supervisor_choice = supervisor_json.get("final_decision")
        supervisor_agreement = prefs.count(supervisor_choice) / len(prefs)
    else:
        supervisor_agreement = consensus_ratio  # fallback

    # 4) Combine them
    stability_raw = (
        0.50 * consensus_ratio +
        0.20 * variance_score +
        0.30 * supervisor_agreement
    )

    stability_score = int(stability_raw * 100)
    return stability_score


In [99]:
def display_stability_score(score):
    print("\n==========================")
    print("📈 DECISION STABILITY SCORE")
    print("==========================")
    print(f"Stability Score: {score}/100")

    if score > 85:
        print("🟢 Very Stable Decision")
    elif score > 70:
        print("🟡 Moderately Stable")
    elif score > 50:
        print("🟠 Mildly Unstable")
    else:
        print("🔴 Unstable — Needs Review")
    print()


In [100]:
score = compute_stability_score(
    trace_output["proposals"],
    trace_output["supervisor"]
)

display_stability_score(score)



📈 DECISION STABILITY SCORE
Stability Score: 79/100
🟡 Moderately Stable



In [101]:
# FEATURE 3 — Option Battle Cards (Side-by-Side Comparison)

def generate_battle_card(proposals, options):
    """
    Creates a comparison table of options across all agents.
    Returns a structured dict suitable for table rendering.
    """

    # Initialize structure
    battle = {
        "options": options,
        "agents": {},
        "winner_votes": {}
    }

    # Count votes
    vote_counts = {opt: 0 for opt in options}

    for p in proposals:
        agent = p["agent_name"]
        battle["agents"][agent] = {}
        preferred = p["preferred_option"]
        vote_counts[preferred] += 1

        # Extract metrics per option
        for opt in options:
            scores = p["scores"].get(opt, {})
            battle["agents"][agent][opt] = scores

    # Determine winner (most votes)
    winner = max(vote_counts, key=vote_counts.get)
    battle["winner_votes"] = vote_counts
    battle["final_winner"] = winner

    return battle


In [102]:
from tabulate import tabulate

def display_battle_card(battle):
    options = battle["options"]
    agents = battle["agents"]

    print("\n======================================")
    print("⚔️  OPTION BATTLE CARD — SIDE BY SIDE")
    print("======================================")

    # Build comparison table
    rows = []
    for agent, result in agents.items():
        row = [agent]
        for opt in options:
            # Format score dict cleanly
            score_text = ", ".join([f"{k}:{v}" for k,v in result[opt].items()])
            row.append(score_text if score_text else "-")
        rows.append(row)

    print(tabulate(rows, headers=["Agent"] + options, tablefmt="fancy_grid"))

    print("\n🗳️  Vote Counts:", battle["winner_votes"])
    print("🏆 Final Winner:", battle["final_winner"])
    print()


In [103]:
battle = generate_battle_card(trace_output["proposals"], trace_output["options"])
display_battle_card(battle)



⚔️  OPTION BATTLE CARD — SIDE BY SIDE
╒═══════════════╤═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╤═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│ Agent         │ Accept the promotion with doubled hours and higher salary                                                                                     │ Stay in current role with less pay and more flexibility                                                                                       │
╞═══════════════╪═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╪═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│ EmotionAgent  │ emotional_relief:0.5, str

In [105]:
# Personality Profiles for Each Agent

agent_personality_profiles = {
    "EmotionAgent": {
        "title": "Emotion Agent",
        "tagline": "Understands feelings, stress, relief, overwhelm.",
        "strengths": [
            "Great at reading emotional signals",
            "Prevents burnout decisions",
            "Balances mental health"
        ],
        "weaknesses": [
            "May ignore logical priorities",
            "Can overprioritize emotional comfort"
        ],
        "personality": "Highly empathetic, stress-aware, sensitive to emotional tone.",
        "color": "#FF6B6B"
    },

    "LogicAgent": {
        "title": "Logic Agent",
        "tagline": "Evaluates time, money, productivity, efficiency.",
        "strengths": [
            "Highly rational decision maker",
            "Great at cost-benefit analysis",
            "Strong at short-term productivity"
        ],
        "weaknesses": [
            "Can ignore emotional toll",
            "Not aware of long-term fulfillment"
        ],
        "personality": "Analytical, structured, objective, data-driven.",
        "color": "#4ECDC4"
    },

    "LongTermAgent": {
        "title": "Long-Term Agent",
        "tagline": "Thinks about future regret, alignment, multi-year outcomes.",
        "strengths": [
            "Prevents impulsive choices",
            "Maximizes long-term gain",
            "Reduces future regret"
        ],
        "weaknesses": [
            "Can ignore present reality",
            "Sometimes overly cautious"
        ],
        "personality": "Forward-thinking, strategic, slow but wise.",
        "color": "#1A535C"
    },

    "ValuesAgent": {
        "title": "Values Agent",
        "tagline": "Checks alignment with core values and life priorities.",
        "strengths": [
            "Ensures value-consistent decisions",
            "Balances all dimensions of life",
            "Strong at ethical judgment"
        ],
        "weaknesses": [
            "May generalize values",
            "Subjective weighting"
        ],
        "personality": "Grounded, reflective, moral compass of the system.",
        "color": "#FFE66D"
    }
}

print("✅ Agent personality profiles loaded.")


✅ Agent personality profiles loaded.


In [106]:
# STEP 2 — Personality Card Renderer

from textwrap import indent

def render_personality_card(agent_name, profile):
    title = profile["title"]
    tagline = profile["tagline"]
    strengths = profile["strengths"]
    weaknesses = profile["weaknesses"]
    personality = profile["personality"]

    print("\n" + "="*70)
    print(f"🎴  {title.upper()}")
    print("="*70)
    print(f"✨ Tagline: {tagline}\n")

    print("💪 Strengths:")
    for s in strengths:
        print(f"   - {s}")

    print("\n⚠️ Weaknesses:")
    for w in weaknesses:
        print(f"   - {w}")

    print("\n🧠 Personality Description:")
    print(indent(personality, "   "))
    print("="*70)


def show_all_agent_personality_cards():
    print("📚 DISPLAYING ALL AGENT PERSONALITY CARDS...\n")
    for agent_name, profile in agent_personality_profiles.items():
        render_personality_card(agent_name, profile)

print("✅ STEP 2 complete: Personality renderer ready.")


✅ STEP 2 complete: Personality renderer ready.


In [107]:
show_all_agent_personality_cards()


📚 DISPLAYING ALL AGENT PERSONALITY CARDS...


🎴  EMOTION AGENT
✨ Tagline: Understands feelings, stress, relief, overwhelm.

💪 Strengths:
   - Great at reading emotional signals
   - Prevents burnout decisions
   - Balances mental health

⚠️ Weaknesses:
   - May ignore logical priorities
   - Can overprioritize emotional comfort

🧠 Personality Description:
   Highly empathetic, stress-aware, sensitive to emotional tone.

🎴  LOGIC AGENT
✨ Tagline: Evaluates time, money, productivity, efficiency.

💪 Strengths:
   - Highly rational decision maker
   - Great at cost-benefit analysis
   - Strong at short-term productivity

⚠️ Weaknesses:
   - Can ignore emotional toll
   - Not aware of long-term fulfillment

🧠 Personality Description:
   Analytical, structured, objective, data-driven.

🎴  LONG-TERM AGENT
✨ Tagline: Thinks about future regret, alignment, multi-year outcomes.

💪 Strengths:
   - Prevents impulsive choices
   - Maximizes long-term gain
   - Reduces future regret

⚠️ Weaknesses:
